# Tables SM

This notebook generates LaTeX tables in the exact `figure` + two `minipage` format shown in your example.

It covers three input files:

- `fit/parameters_goodfit_norm_SRF_micro_bacilla.csv`
- `fit/parameters_goodfit_norm_core_SRF_micro_bacilla.csv`
- `fit/parameters_goodfit_norm_sector_SRF_micro_bacilla.csv`

The exported `.tex` files are written to `analysis/tables_sm/` and are numbered in this fixed order:

1. `metaG` all
2. `metaT` all
3. external table slot
4. external table slot
5. `metaG` all, raw-count placeholder
6. `metaT` all, raw-count placeholder
7. `metaG` core
8. `metaG` non-core
9. `metaT` core
10. `metaT` non-core
11. `metaG` sector Q
12. `metaG` sector R
13. `metaT` sector Q
14. `metaT` sector R

In [10]:
from math import ceil
from pathlib import Path

import pandas as pd
from IPython.display import display

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / 'fit').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing the fit/ directory.')

FIT_FILES = {
    'all': PROJECT_ROOT / 'fit' / 'parameters_goodfit_norm_SRF_micro_bacilla.csv',
    'core': PROJECT_ROOT / 'fit' / 'parameters_goodfit_norm_core_SRF_micro_bacilla.csv',
    'sector': PROJECT_ROOT / 'fit' / 'parameters_goodfit_norm_sector_SRF_micro_bacilla.csv',
}

OUTPUT_SPECS = [
    {
        'table_kind': 'all',
        'data': 'metaG',
        'group': None,
        'table_number': 1,
        'output_file': 'figure_sm_all_metaG.tex',
    },
    {
        'table_kind': 'all',
        'data': 'metaT',
        'group': None,
        'table_number': 2,
        'output_file': 'figure_sm_all_metaT.tex',
    },
    {
        'table_kind': 'all_raw',
        'source_kind': 'all',
        'data': 'metaG',
        'group': None,
        'table_number': 5,
        'output_file': 'figure_sm_all_raw_metaG.tex',
    },
    {
        'table_kind': 'all_raw',
        'source_kind': 'all',
        'data': 'metaT',
        'group': None,
        'table_number': 6,
        'output_file': 'figure_sm_all_raw_metaT.tex',
    },
    {
        'table_kind': 'core',
        'data': 'metaG',
        'group': 'core',
        'table_number': 7,
        'output_file': 'figure_sm_core_metaG_core.tex',
    },
    {
        'table_kind': 'core',
        'data': 'metaG',
        'group': 'non_core',
        'table_number': 8,
        'output_file': 'figure_sm_core_metaG_non_core.tex',
    },
    {
        'table_kind': 'core',
        'data': 'metaT',
        'group': 'core',
        'table_number': 9,
        'output_file': 'figure_sm_core_metaT_core.tex',
    },
    {
        'table_kind': 'core',
        'data': 'metaT',
        'group': 'non_core',
        'table_number': 10,
        'output_file': 'figure_sm_core_metaT_non_core.tex',
    },
    {
        'table_kind': 'sector',
        'data': 'metaG',
        'group': 'Q',
        'table_number': 11,
        'output_file': 'figure_sm_sector_metaG_Q.tex',
    },
    {
        'table_kind': 'sector',
        'data': 'metaG',
        'group': 'R',
        'table_number': 12,
        'output_file': 'figure_sm_sector_metaG_R.tex',
    },
    {
        'table_kind': 'sector',
        'data': 'metaT',
        'group': 'Q',
        'table_number': 13,
        'output_file': 'figure_sm_sector_metaT_Q.tex',
    },
    {
        'table_kind': 'sector',
        'data': 'metaT',
        'group': 'R',
        'table_number': 14,
        'output_file': 'figure_sm_sector_metaT_R.tex',
    },
]

OUTPUT_DIR = PROJECT_ROOT / 'analysis' / 'tables_sm'
OUTPUT_DIR.mkdir(exist_ok=True)

FLOAT_POSITION = 'htbp!'
MINIPAGE_WIDTH = '0.48\\linewidth'
MINIPAGE_GAP = '0.02\\linewidth'
TABLE_FONT_SIZE = '\\footnotesize'
TABULAR_ALIGNMENT = 'llllll'
REJECTED_STYLE = 'bold'  # valid values: 'bold', 'symbol', 'both', 'none'
SHOW_FULL_LATEX = False
EQUATION_REFERENCE = '\\eqref{app:eq:fit_distrib_renorm}'

FIT_FILES

{'all': PosixPath('/Users/epigani/Library/CloudStorage/GoogleDrive-emanuele.pigani.1@unipd.it/.shortcut-targets-by-id/17Sn-Ra2REh5B86l96gE_Smchh22WCsgo/PhD-Emanuele Pigani/finished_projects/Pigani et al 2026/diatom-metaG-metaT-scaling/fit/parameters_goodfit_norm_SRF_micro_bacilla.csv'),
 'core': PosixPath('/Users/epigani/Library/CloudStorage/GoogleDrive-emanuele.pigani.1@unipd.it/.shortcut-targets-by-id/17Sn-Ra2REh5B86l96gE_Smchh22WCsgo/PhD-Emanuele Pigani/finished_projects/Pigani et al 2026/diatom-metaG-metaT-scaling/fit/parameters_goodfit_norm_core_SRF_micro_bacilla.csv'),
 'sector': PosixPath('/Users/epigani/Library/CloudStorage/GoogleDrive-emanuele.pigani.1@unipd.it/.shortcut-targets-by-id/17Sn-Ra2REh5B86l96gE_Smchh22WCsgo/PhD-Emanuele Pigani/finished_projects/Pigani et al 2026/diatom-metaG-metaT-scaling/fit/parameters_goodfit_norm_sector_SRF_micro_bacilla.csv')}

In [11]:
def detect_group_column(df):
    for candidate in ('subset', 'sector'):
        if candidate in df.columns:
            return candidate
    return None


def metric_suffix(data_name):
    return {'metaG': 'G', 'metaT': 'T'}.get(data_name, data_name)


def safe_group_token(group_value):
    if group_value is None:
        return 'all'
    return str(group_value).replace(' ', '_').replace('-', '_')


def group_caption_text(table_kind, group_value):
    if table_kind in ('all', 'all_raw'):
        return 'all genes'
    if table_kind == 'core':
        mapping = {
            'core': 'core unigenes',
            'non_core': 'non-core unigenes',
        }
        return mapping.get(str(group_value), f'{group_value} unigenes')
    if table_kind == 'sector':
        return f'sector {group_value} unigenes'
    raise ValueError(f'Unsupported table kind: {table_kind}')


def format_station(value):
    if pd.isna(value):
        return ''
    return f'{int(value):03d}'


def format_fixed(value, digits=2):
    if pd.isna(value):
        return '--'
    return f'{value:.{digits}f}'


def format_scientific(value, digits=2):
    if pd.isna(value):
        return '--'
    return f'{value:.{digits}e}'


def format_k(value):
    if pd.isna(value):
        return '--'
    if abs(value) < 1e-2 or abs(value) >= 1e3:
        return f'{value:.2e}'
    return f'{value:.2f}'


def format_p_cvm(value, rejected, style='bold'):
    text = format_scientific(value, digits=2)
    if text == '--':
        return text

    is_rejected = bool(pd.notna(rejected) and int(rejected) == 1)
    if not is_rejected or style == 'none':
        return text
    if style == 'bold':
        return rf'\textbf{{{text}}}'
    if style == 'symbol':
        return text + r'$^{\dagger}$'
    if style == 'both':
        return rf'\textbf{{{text}}}$^{{\dagger}}$'
    raise ValueError(f'Unknown rejected style: {style}')


def row_to_strings(row, rejected_style='bold'):
    return [
        format_station(row.station),
        format_scientific(row.nmin, digits=2),
        format_fixed(row.zeta, digits=2),
        format_fixed(row.alpha, digits=2),
        format_k(row.k),
        format_p_cvm(row.p_CvM, row.rejected, style=rejected_style),
    ]


def blank_row():
    return ['', '', '', '', '', '']


def split_rows_evenly(rows):
    midpoint = ceil(len(rows) / 2)
    left_rows = rows[:midpoint]
    right_rows = rows[midpoint:]

    while len(right_rows) < len(left_rows):
        right_rows.append(blank_row())

    return left_rows, right_rows


def build_tabular_block(rows, data_name):
    suffix = metric_suffix(data_name)
    header_line_1 = (
        rf' & $n_{{\min}}$ & $\zeta_{suffix}$ & $\alpha_{suffix}$ & $k_{suffix}$ & CvM $p$ \\'
    )
    header_line_2 = r'station &  &  &  &  &  \\'

    lines = [
        rf'\begin{{tabular}}{{{TABULAR_ALIGNMENT}}}',
        r'\toprule',
        header_line_1,
        header_line_2,
        r'\midrule',
    ]

    for values in rows:
        lines.append('            ' + ' & '.join(values) + r'\\')

    lines.extend([
        r'\bottomrule',
        r'\end{tabular}',
    ])
    return lines


def build_caption(table_number, table_kind, data_name, group_value):
    body = (
        f'Fit of {EQUATION_REFERENCE} applied to '
        f'{group_caption_text(table_kind, group_value)} in {data_name} samples. '
        r'CvM $p$ means Cram\'er-von Mises $p$-values, bold text indicates rejected fits.'
    )
    return rf'\caption*{{\textbf{{Table~S{table_number}.}} {body}}}'


def build_label(table_kind, data_name, group_value):
    token = safe_group_token(group_value)
    if table_kind == 'all':
        return f'subtab:app:all:{data_name}'
    if table_kind == 'all_raw':
        return f'subtab:app:all_raw:{data_name}'
    return f'subtab:app:{table_kind}:{data_name}:{token}'


def prepare_group_frame(csv_path, data_name, group_value=None):
    df = pd.read_csv(csv_path)
    group_column = detect_group_column(df)

    mask = df['data'] == data_name
    if group_column is not None and group_value is not None:
        mask &= df[group_column].astype(str) == str(group_value)

    group_df = df.loc[
        mask,
        ['station', 'nmin', 'zeta', 'alpha', 'logk', 'p_CvM', 'rejected']
    ].copy()
    group_df['k'] = 10 ** group_df['logk']
    group_df = group_df.sort_values('station').reset_index(drop=True)
    return group_df, group_column


def build_figure_latex(group_df, table_kind, data_name, group_value, table_number, rejected_style='bold'):
    rows = [row_to_strings(row, rejected_style=rejected_style) for row in group_df.itertuples(index=False)]
    left_rows, right_rows = split_rows_evenly(rows)

    left_block = build_tabular_block(left_rows, data_name)
    right_block = build_tabular_block(right_rows, data_name)

    lines = [
        rf'\begin{{figure}}[{FLOAT_POSITION}]',
        r'    \centering',
        rf'    {{{TABLE_FONT_SIZE}',
        rf'    \begin{{minipage}}[t]{{{MINIPAGE_WIDTH}}}',
        r'        \centering',
    ]
    lines.extend('        ' + line for line in left_block)
    lines.extend([
        r'    \end{minipage}',
        rf'    \hspace{{{MINIPAGE_GAP}}}',
        rf'    \begin{{minipage}}[t]{{{MINIPAGE_WIDTH}}}',
        r'        \centering',
    ])
    lines.extend('        ' + line for line in right_block)
    lines.extend([
        r'    \end{minipage}',
        r'    }',
        '    ' + build_caption(table_number, table_kind, data_name, group_value),
        rf'    \label{{{build_label(table_kind, data_name, group_value)}}}',
        r'\end{figure}',
    ])
    return '\n'.join(lines)


def export_exact_format_tables(rejected_style='bold', show_full_latex=False):
    exports = []
    latex_tables = {}
    input_lines = []

    for spec in OUTPUT_SPECS:
        table_kind = spec['table_kind']
        source_kind = spec.get('source_kind', table_kind)
        data_name = spec['data']
        group_value = spec['group']
        table_number = spec['table_number']
        output_file = spec['output_file']

        group_df, group_column = prepare_group_frame(FIT_FILES[source_kind], data_name, group_value)
        if group_df.empty:
            raise ValueError(
                f'No rows found for table_kind={table_kind}, source_kind={source_kind}, data={data_name}, group={group_value}'
            )

        latex = build_figure_latex(
            group_df,
            table_kind=table_kind,
            data_name=data_name,
            group_value=group_value,
            table_number=table_number,
            rejected_style=rejected_style,
        )

        output_path = OUTPUT_DIR / output_file
        output_path.write_text(latex)
        latex_tables[(table_kind, data_name, safe_group_token(group_value))] = latex
        input_lines.append(rf'\input{{tables_sm/{output_file}}}')
        if output_file == 'figure_sm_all_metaT.tex':
            input_lines.append('% Insert here the two unrelated external tables (Table~S3 and Table~S4)')

        exports.append({
            'table_number': table_number,
            'table_kind': table_kind,
            'data': data_name,
            'group': safe_group_token(group_value),
            'rows': len(group_df),
            'missing_fit_rows': int(group_df[['zeta', 'alpha', 'logk', 'p_CvM']].isna().any(axis=1).sum()),
            'rejected_rows': int((group_df['rejected'].fillna(0) == 1).sum()),
            'output_file': output_path.relative_to(PROJECT_ROOT).as_posix(),
        })

        print(f'Saved {output_path.relative_to(PROJECT_ROOT)}')
        if show_full_latex:
            print(latex)
        else:
            print('\n'.join(latex.splitlines()[:22]))
            print('...')
        print()

    return pd.DataFrame(exports), latex_tables, input_lines


def preview_group_table(table_kind, data_name, group_value=None, rejected_style='bold'):
    group_df, group_column = prepare_group_frame(FIT_FILES[table_kind], data_name, group_value)
    preview = pd.DataFrame({
        'station': group_df['station'].map(format_station),
        'nmin': group_df['nmin'].map(format_scientific),
        'zeta': group_df['zeta'].map(format_fixed),
        'alpha': group_df['alpha'].map(format_fixed),
        'k': group_df['k'].map(format_k),
        'p_CvM': [
            format_p_cvm(value, rejected, style=rejected_style)
            for value, rejected in zip(group_df['p_CvM'], group_df['rejected'])
        ],
    })
    return preview, group_column


In [12]:
summary_df, latex_tables, input_lines = export_exact_format_tables(
    rejected_style=REJECTED_STYLE,
    show_full_latex=SHOW_FULL_LATEX,
)

display(summary_df)

sample_key = ('all', 'metaG', 'all')
sample_preview_df, sample_group_column = preview_group_table(
    table_kind='all',
    data_name='metaG',
    group_value=None,
    rejected_style=REJECTED_STYLE,
)
display(sample_preview_df.head(12))

print('Overleaf inputs:')
for line in input_lines:
    print(line)

print()
print(f'Use latex_tables[{sample_key!r}] to inspect the full LaTeX string in the notebook.')


Saved analysis/tables_sm/figure_sm_all_metaG.tex
\begin{figure}[htbp!]
    \centering
    {\footnotesize
    \begin{minipage}[t]{0.48\linewidth}
        \centering
        \begin{tabular}{llllll}
        \toprule
         & $n_{\min}$ & $\zeta_G$ & $\alpha_G$ & $k_G$ & CvM $p$ \\
        station &  &  &  &  &  \\
        \midrule
                    007 & 5.38e-04 & 2.59 & 1.44 & 0.03 & 6.50e-03\\
                    011 & 2.51e-03 & 2.69 & 0.76 & 0.02 & 6.90e-01\\
                    018 & 1.53e-03 & 6.02 & 1.21 & 7.81e-03 & 7.94e-01\\
                    022 & 9.27e-04 & 2.48 & 1.22 & 0.02 & \textbf{2.50e-04}\\
                    023 & 1.26e-03 & 3.90 & 1.18 & 0.01 & 5.03e-01\\
                    025 & 8.33e-04 & 5.38 & 0.37 & 5.98e-03 & 3.01e-01\\
                    026 & 1.73e-03 & 5.99 & 0.74 & 6.51e-03 & 1.65e-02\\
                    030 & 1.29e-03 & 5.85 & 1.31 & 0.01 & 7.80e-02\\
                    036 & 4.43e-04 & 2.72 & -0.00 & 7.47e-03 & 1.50e-03\\
                    0

,table_number,table_kind,data,group,rows,missing_fit_rows,rejected_rows,output_file
0,1,all,metaG,all,77,0,45,analysis/tables_sm/figure_sm_all_metaG.tex
1,2,all,metaT,all,77,0,35,analysis/tables_sm/figure_sm_all_metaT.tex
2,5,all_raw,metaG,all,77,0,45,analysis/tables_sm/figure_sm_all_raw_metaG.tex
3,6,all_raw,metaT,all,77,0,35,analysis/tables_sm/figure_sm_all_raw_metaT.tex
4,7,core,metaG,core,77,0,40,analysis/tables_sm/figure_sm_core_metaG_core.tex
5,8,core,metaG,non_core,77,8,31,analysis/tables_sm/figure_sm_core_metaG_non_co...
6,9,core,metaT,core,77,0,20,analysis/tables_sm/figure_sm_core_metaT_core.tex
7,10,core,metaT,non_core,77,9,23,analysis/tables_sm/figure_sm_core_metaT_non_co...
8,11,sector,metaG,Q,77,12,29,analysis/tables_sm/figure_sm_sector_metaG_Q.tex
9,12,sector,metaG,R,77,3,37,analysis/tables_sm/figure_sm_sector_metaG_R.tex


,station,nmin,zeta,alpha,k,p_CvM
0,007,5.38e-04,2.59,1.44,0.03,6.50e-03
1,011,2.51e-03,2.69,0.76,0.02,6.90e-01
2,018,1.53e-03,6.02,1.21,7.81e-03,7.94e-01
3,022,9.27e-04,2.48,1.22,0.02,\textbf{2.50e-04}
4,023,1.26e-03,3.90,1.18,0.01,5.03e-01
5,025,8.33e-04,5.38,0.37,5.98e-03,3.01e-01
6,026,1.73e-03,5.99,0.74,6.51e-03,1.65e-02
7,030,1.29e-03,5.85,1.31,0.01,7.80e-02
8,036,4.43e-04,2.72,-0.00,7.47e-03,1.50e-03
9,038,2.43e-03,6.67,0.97,6.98e-03,8.43e-02


Overleaf inputs:
\input{tables_sm/figure_sm_all_metaG.tex}
\input{tables_sm/figure_sm_all_metaT.tex}
% Insert here the two unrelated external tables (Table~S3 and Table~S4)
\input{tables_sm/figure_sm_all_raw_metaG.tex}
\input{tables_sm/figure_sm_all_raw_metaT.tex}
\input{tables_sm/figure_sm_core_metaG_core.tex}
\input{tables_sm/figure_sm_core_metaG_non_core.tex}
\input{tables_sm/figure_sm_core_metaT_core.tex}
\input{tables_sm/figure_sm_core_metaT_non_core.tex}
\input{tables_sm/figure_sm_sector_metaG_Q.tex}
\input{tables_sm/figure_sm_sector_metaG_R.tex}
\input{tables_sm/figure_sm_sector_metaT_Q.tex}
\input{tables_sm/figure_sm_sector_metaT_R.tex}

Use latex_tables[('all', 'metaG', 'all')] to inspect the full LaTeX string in the notebook.
